# Experiment 010: Llama 3.2-1B-Instruct Full Run (5K steps)

Full training run following the successful 009 smoke test (100% operation routing at 250 steps).

| Parameter | Value |
|-----------|-------|
| Model | meta-llama/Llama-3.2-1B-Instruct (1.24B params, all trainable) |
| Data | 100K (v2 generator: formulaic + hard negatives + edge cases) |
| Distribution | 30% ADD / 30% SUB / 15% BETWEEN / 25% NO_ROUTE |
| Steps | 5,000 |
| Effective batch | 16 (2 x 8) |
| Epochs | ~0.84 (sub-epoch to reduce catastrophic forgetting risk) |
| Precision | bf16 (L4 native) |
| Checkpoints | Every 500 steps (10 checkpoints, keep last 8) |

**Runtime:** L4 GPU. ~2-3 hours.

**Goal:** Fix argument precision (minute extraction, duration values) and eliminate trailing token repetition observed in 009.

In [ ]:
!pip install -q transformers datasets accelerate torch tensorboard bitsandbytes

In [ ]:
import torch, os, subprocess, math, json, numpy as np
assert torch.cuda.is_available(), 'No GPU!'
assert torch.cuda.is_bf16_supported(), 'Need bf16 - use L4, not T4'
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('Authenticated')
except Exception:
    login()

## 1. Generate 100K Training Data

In [ ]:
if not os.path.exists('tagzeit-gemma-2b'):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git
else:
    !cd tagzeit-gemma-2b && git pull
!cd tagzeit-gemma-2b && npm install --silent
print('Ready')

In [ ]:
# Extract hard negatives
!cd tagzeit-gemma-2b && python tools/extract_hard_negatives.py \
    --output data/hard_negatives/complex_tempqa_noroute.jsonl --count 7000

# Generate 100K samples
!cd tagzeit-gemma-2b && node experiments/temporal-tagzeit/src/generator_route.js \
    --count 100000 \
    --output data/train/train_routed_010.jsonl \
    --eval data/eval/eval_routed_010.jsonl \
    --hard-negatives data/hard_negatives/complex_tempqa_noroute.jsonl

In [ ]:
# Format for Llama chat template
!cd tagzeit-gemma-2b && python tools/format_for_model.py \
    --model_id meta-llama/Llama-3.2-1B-Instruct \
    --input data/train/train_routed_010.jsonl \
    --output data/train/train_routed_010_llama1b.jsonl \
    --eval_input data/eval/eval_routed_010.jsonl \
    --eval_output data/eval/eval_routed_010_llama1b.jsonl

TRAIN_FILE = 'tagzeit-gemma-2b/data/train/train_routed_010_llama1b.jsonl'
EVAL_FILE  = 'tagzeit-gemma-2b/data/eval/eval_routed_010_llama1b.jsonl'

for f in [TRAIN_FILE, EVAL_FILE]:
    n = int(subprocess.check_output(['wc', '-l', f]).split()[0])
    print(f'{os.path.basename(f)}: {n} samples')

# Spot-check
with open(TRAIN_FILE) as fh:
    sample = json.loads(fh.readline())
    print(f'\nSample:\n{sample["text"][:300]}')

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/tagzeit-010'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to {OUTPUT_DIR}')

## 2. Load Model + Domain Tokens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

ROUTE_TOKENS = [
    '[ROUTE]', '[/ROUTE]', '[NO_ROUTE]',
    '[ROUTE_TIME_ADD]', '[ROUTE_TIME_SUB]', '[ROUTE_DURATION_BETWEEN]',
    '[ROUTE_CALENDAR_SHIFT]', '[ROUTE_TIMEZONE_CONVERT]',
    '[HEAD_TIME]', '[HEAD_DURATION]', '[HEAD_DATE]',
    '[REF_1]', '[REF_2]', '[REF_3]',
]
ROUTE_TOKENS += [f'[ARG_HOUR_{str(h).zfill(2)}]' for h in range(24)]
ROUTE_TOKENS += [f'[ARG_MIN_{str(m).zfill(2)}]' for m in range(60)]
ROUTE_TOKENS += ['[ARG_MON]', '[ARG_TUE]', '[ARG_WED]', '[ARG_THU]', '[ARG_FRI]', '[ARG_SAT]', '[ARG_SUN]']

num_added = tokenizer.add_tokens(ROUTE_TOKENS)
model.resize_token_embeddings(len(tokenizer))

embed_dim = model.config.hidden_size
with torch.no_grad():
    new_start = len(tokenizer) - num_added
    for i in range(num_added):
        freq = np.array([1.0 / (10000 ** (2 * j / embed_dim)) for j in range(embed_dim)])
        phase = (i + 1) * freq
        init_vec = np.sin(phase) * 0.02
        model.get_input_embeddings().weight[new_start + i] = torch.tensor(init_vec, dtype=torch.bfloat16)

print(f'Loaded: {sum(p.numel() for p in model.parameters()):,} params, {num_added} domain tokens')
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 3. Tokenize Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'eval': EVAL_FILE})

def tokenize_fn(examples):
    result = tokenizer(examples['text'], truncation=True, max_length=384, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

n_train = len(tokenized['train'])
eff_batch = 2 * 8
epochs = (5000 * eff_batch) / n_train
print(f'Train: {n_train}, Eval: {len(tokenized["eval"])}')
print(f'5K steps x {eff_batch} batch = {5000*eff_batch:,} samples = {epochs:.2f} epochs')

## 4. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=5000,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=200,
    weight_decay=0.01,
    optim='adamw_8bit',
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=8,
    logging_steps=25,
    logging_dir=f'{OUTPUT_DIR}/logs',
    report_to='tensorboard',
    dataloader_num_workers=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['eval'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    processing_class=tokenizer,
)

print(f'Ready: 5K steps, checkpoints every 500, bf16')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
train_result = trainer.train()

print(f'\nTrain loss: {train_result.training_loss:.4f}')
print(f'Runtime: {train_result.metrics["train_runtime"]:.0f}s ({train_result.metrics["train_runtime"]/60:.0f} min)')

eval_results = trainer.evaluate()
print(f'Eval loss: {eval_results["eval_loss"]:.4f}')
print(f'Eval perplexity: {math.exp(eval_results["eval_loss"]):.2f}')

## 5. Inference Test

In [ ]:
test_prompts = [
    ('What time is it 1 minute after 23:59?', 'ADD'),
    ('What time is it 15 minutes after 12:00?', 'ADD'),
    ('What time was it 30 minutes before 00:15?', 'SUB'),
    ('What time was it 1 hour before 01:00?', 'SUB'),
    ('How much time is there between 09:00 and 17:30?', 'BETWEEN'),
    ('How long is it from 23:45 to 00:15?', 'BETWEEN'),
    ('The meeting starts at 14:30 and lasts 45 minutes. When does it end?', 'ADD'),
    ('I arrived at work at 09:15. The commute was 25 minutes. When did I leave home?', 'SUB'),
    ('What is 42 + 18?', 'NO_ROUTE'),
    ('When was The Secret Garden published?', 'NO_ROUTE'),
    ('What timezone is Tokyo in?', 'NO_ROUTE'),
    ('How many calories are in an apple?', 'NO_ROUTE'),
]

check_map = {
    'ADD': '[ROUTE_TIME_ADD]',
    'SUB': '[ROUTE_TIME_SUB]',
    'BETWEEN': '[ROUTE_DURATION_BETWEEN]',
    'NO_ROUTE': '[NO_ROUTE]',
}

model.eval()
correct = 0

for prompt, expected in test_prompts:
    messages = [{'role': 'user', 'content': prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    response = response.split('<|eot_id|>')[0].strip()

    got_right = check_map[expected] in response
    if got_right:
        correct += 1

    print(f"{'PASS' if got_right else 'FAIL'} [{expected}] {prompt}")
    print(f'   {response[:150]}')
    print()

print('=' * 50)
print(f'Routing: {correct}/{len(test_prompts)} ({correct/len(test_prompts)*100:.0f}%)')

## 6. Save Best Model

In [ ]:
trainer.save_model(f'{OUTPUT_DIR}/best')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/best')
print(f'Best model saved to {OUTPUT_DIR}/best')

import glob
checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
print(f'\nCheckpoints ({len(checkpoints)}):')
for cp in checkpoints:
    print(f'  {os.path.basename(cp)}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs

---
## Resume from Checkpoint

If Colab disconnects, re-run cells 1-4 (setup through tokenize), then:

In [ ]:
# Re-run cells 1-4 first, then:
# train_result = trainer.train(resume_from_checkpoint=True)